# Track A - run training and view results
This notebook calls `arterylens__train.py` (the real code lives there) and displays every output file.

In [ ]:
# 1) Run training (takes ~1-3 minutes). Same as: !python arterylens__train.py --data "..." 
DATA = "../data/extention of Z-Alizadeh sani dataset.xlsx"   # change to your path
%run arterylens__train.py --data "{DATA}" --repeats 3

In [ ]:
# 2) Reload the modules whenever you edit arterylens__train.py (no kernel restart needed)
import importlib, arterylens__train as train
importlib.reload(train)
res = train.run(DATA, n_repeats=3)   # returns dict with cv table, best models, X, y
res["best"]

In [ ]:
# 3) All models x all targets (mean and std over repeated CV)
import pandas as pd
pd.read_csv("reports/cv_metrics_all_models.csv")

In [ ]:
# 4) Plots saved by arterylens__train.py
from IPython.display import Image, display
for f in ["roc_calibration.png", "confusion_matrices.png",
          "shap_summary_CAD.png", "shap_summary_LAD.png", "shap_summary_LCX.png", "shap_summary_RCA.png"]:
    print(f); display(Image("reports/" + f))

In [ ]:
# 5) Training log + best-model JSON
print(open("reports/training.log").read())
import json; print(json.dumps(json.load(open("reports/metrics.json"))["shap_top_features"]["LAD"], indent=2))

In [ ]:
# 6) Try a prediction with a saved model (this is exactly what the backend will do)
import joblib
X = res["X"]
patient = X.iloc[[0]]                       # first patient as a 1-row DataFrame
for t in ["CAD", "LAD", "LCX", "RCA"]:
    m = joblib.load(f"models/model_{t.lower()}.joblib")
    print(t, round(m.predict_proba(patient)[0, 1], 3))